In [1]:
import os; os.environ['WORKDIR'] = "/home/choij/workspace/ChargedHiggsAnalysis"
import shutil
import sys; sys.path.insert(0, os.environ['WORKDIR'])
import pandas as pd

In [2]:
model_info = pd.read_csv(f"{os.environ['WORKDIR']}/MetaInfo/models.csv")
model_info.set_index(['signal', 'background'], inplace=True)

In [3]:
model_info

optim  initial_lr      scheduler  ksprob  \
signal         background                                                
MHc-160_MA-155 VV              Adam       0.008  ExponentialLR   0.819   
MHc-130_MA-55  TTLL_powheg  RMSprop       0.001  ExponentialLR   0.566   
MHc-130_MA-15  VV              Adam       0.005         StepLR   0.595   
MHc-100_MA-15  TTLL_powheg     Adam       0.002  ExponentialLR   0.718   
               VV              Adam       0.002  ExponentialLR   0.434   
MHc-160_MA-155 TTLL_powheg     Adam       0.005         StepLR   0.453   
MHc-160_MA-15  VV              Adam       0.002  ExponentialLR   0.400   
MHc-130_MA-125 TTLL_powheg     Adam       0.001  ExponentialLR   0.228   
MHc-70_MA-15   VV              Adam       0.002  ExponentialLR   0.344   
MHc-160_MA-15  TTLL_powheg     Adam       0.001  ExponentialLR   0.687   
MHc-70_MA-40   TTLL_powheg     Adam       0.005         StepLR   0.238   
MHc-160_MA-120 VV              Adam       0.001         StepLR   0.265   
MHc-100_MA-95  TTLL_powheg     Adam       0.002  ExponentialLR   0.278   
MHc-130_MA-15  TTLL_powheg     Adam       0.001         StepLR   0.654   
MHc-160_MA-85  TTLL_powheg     Adam       0.001  ExponentialLR   0.204   
MHc-70_MA-40   VV              Adam       0.001         StepLR   0.565   
MHc-70_MA-65   VV              Adam       0.001  ExponentialLR   0.394   
MHc-100_MA-60  TTLL_powheg     Adam       0.005  ExponentialLR   0.464   
MHc-70_MA-15   TTLL_powheg     Adam       0.001         StepLR   0.600   
MHc-100_MA-95  VV              Adam       0.002  ExponentialLR   0.942   
MHc-100_MA-60  VV              Adam       0.001  ExponentialLR   0.231   
MHc-130_MA-125 VV              Adam       0.005  ExponentialLR   0.729   
MHc-130_MA-90  VV              Adam       0.008  ExponentialLR   0.967   
MHc-160_MA-120 TTLL_powheg     Adam       0.002  ExponentialLR   0.278   
MHc-130_MA-55  VV              Adam       0.001         StepLR   0.434   
MHc-70_MA-65   TTLL_powheg     Adam       0.008         StepLR   0.400   
MHc-160_MA-85  VV              Adam       0.002  ExponentialLR   0.762   
MHc-130_MA-90  TTLL_powheg     Adam       0.005         StepLR   0.240   

                            train_auc  valid_auc  test_auc  
signal         background                                   
MHc-160_MA-155 VV            0.723363   0.724905  0.723482  
MHc-130_MA-55  TTLL_powheg   0.668581   0.670723  0.669798  
MHc-130_MA-15  VV            0.933849   0.935645  0.932960  
MHc-100_MA-15  TTLL_powheg   0.957156   0.956423  0.954223  
               VV            0.989645   0.989291  0.988558  
MHc-160_MA-155 TTLL_powheg   0.965641   0.963208  0.963392  
MHc-160_MA-15  VV            0.986975   0.986708  0.986471  
MHc-130_MA-125 TTLL_powheg   0.912443   0.903184  0.901548  
MHc-70_MA-15   VV            0.994831   0.994597  0.994190  
MHc-160_MA-15  TTLL_powheg   0.961597   0.958117  0.960313  
MHc-70_MA-40   TTLL_powheg   0.971070   0.967760  0.968138  
MHc-160_MA-120 VV            0.962621   0.957298  0.957528  
MHc-100_MA-95  TTLL_powheg   0.874953   0.867135  0.865311  
MHc-130_MA-15  TTLL_powheg   0.961363   0.961112  0.960163  
MHc-160_MA-85  TTLL_powheg   0.849004   0.825968  0.826020  
MHc-70_MA-40   VV            0.981082   0.980889  0.979430  
MHc-70_MA-65   VV            0.974522   0.973654  0.973712  
MHc-100_MA-60  TTLL_powheg   0.886606   0.881521  0.877745  
MHc-70_MA-15   TTLL_powheg   0.968864   0.966513  0.967851  
MHc-100_MA-95  VV            0.971994   0.971457  0.970818  
MHc-100_MA-60  VV            0.977816   0.977833  0.976946  
MHc-130_MA-125 VV            0.975703   0.973314  0.972353  
MHc-130_MA-90  VV            0.966754   0.965629  0.965492  
MHc-160_MA-120 TTLL_powheg   0.877701   0.864468  0.860858  
MHc-130_MA-55  VV            0.975095   0.973708  0.973881  
MHc-70_MA-65   TTLL_powheg   0.869813   0.856535  0.856155  
MHc-160_MA-85  VV            0.955145   0.954039  0.953427  
MHc-130_MA-90  TTLL_powheg   0.930292   

In [4]:
def getSuffix(masspoint, background):
    classifier = (masspoint, background)
    optim = model_info.loc[classifier, 'optim']
    initial_lr = model_info.loc[classifier, 'initial_lr']
    scheduler = model_info.loc[classifier, 'scheduler']
    
    return f"ParticleNet_nhidden-128_{optim}_initial_lr-{str(initial_lr).replace('.', 'p')}_{scheduler}_nbatch-1024"

In [5]:
MASSPOINTs = ["MHc-70_MA-15", "MHc-70_MA-40", "MHc-70_MA-65",
              "MHc-100_MA-15", "MHc-100_MA-60", "MHc-100_MA-95",
              "MHc-130_MA-15", "MHc-130_MA-55", "MHc-130_MA-90", "MHc-130_MA-125",
              "MHc-160_MA-15", "MHc-160_MA-85", "MHc-160_MA-120", "MHc-160_MA-155"]
BACKGROUNDs = ["TTLL_powheg", "VV"]

from itertools import product
for masspoint, background in product(MASSPOINTs, BACKGROUNDs):
    prefix = f"{os.environ['WORKDIR']}/triLepRegion/plots/All/{masspoint}_vs_{background}"
    target = f"{os.environ['WORKDIR']}/triLepRegion/plots/results/{masspoint}_vs_{background}"
    if not os.path.exists(target):
        os.makedirs(target)
        
    suffix = getSuffix(masspoint, background)
    shutil.copy(f"{prefix}/roc-{suffix}.png", f"{target}/roc-{suffix}.png")
    shutil.copy(f"{prefix}/training-{suffix}.png", f"{target}/training-{suffix}.png")
    
    prefix = f"{os.environ['WORKDIR']}/triLepRegion/ROOT/All/{masspoint}_vs_{background}"
    shutil.copy(f"{prefix}/{suffix}.root", f"{target}/{suffix}.root")